# Stage 7-1. Multiclass 전체 실험 — MLP + CNN 비교

이 노트북은 MNIST 10-class 분류 과제의 전체 실험 파이프라인을 실습한다.

| 항목 | 값 |
|---|---|
| 과제 | 10개 digit class 분류 |
| target 변환 | digit label → one-hot, shape `(10,)` |
| output_dim | 10 |
| loss | cross_entropy |
| metric | accuracy |
| prediction_mode | argmax |

**학습 목표**
1. `Experiment(config)`로 MLP / CNN 파이프라인을 조립하고 학습을 실행한다.
2. 학습 곡선과 예측 grid를 시각화한다.
3. MLP vs CNN 성능을 정량 비교한다.
4. 저장된 checkpoint를 불러와 평가를 재현한다.
5. CLI 실행 명령을 확인한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt
import os

try:
    import cupy
    _BACKEND = f"CuPy {cupy.__version__}"
except ImportError:
    _BACKEND = "NumPy (CuPy 미설치 — fallback)"

print(f"백엔드: {_BACKEND}")

from src.core.experiment import Experiment
from src.core import checkpoints
from src.data.mnist import MnistDataset
from src.task import get_task_spec

DATASET_DIR = "/mnt/d/datasets/mnist"
OUTPUT_DIR  = "../../outputs/multiclass"
TASK        = "multiclass"

print(f"\ntask: {TASK}")
spec = get_task_spec(TASK)
print(f"  output_dim     : {spec['output_dim']}")
print(f"  prediction_mode: {spec['prediction_mode']}")

## 1. MLP 실험

### 1.1. MLP 학습

In [ ]:
config_mlp = {
    "dataset_dir": DATASET_DIR,
    "task":        TASK,
    "model":       "mlp",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  10,
    "lr":          0.01,
}

exp_mlp = Experiment(config_mlp)
print(f"MLP 파라미터 수: {sum(p.size for p in exp_mlp.model.params):,}")
print("학습 시작 (10 epochs)...")
logs_mlp = exp_mlp.run()

print()
print(f"{'epoch':>5} | {'train loss':>10} {'train acc':>10} | {'test loss':>10} {'test acc':>10}")
print("-" * 55)
for log in logs_mlp:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>10.4f} {log['train']['metric']:>10.4f} | "
          f"{log['test']['loss']:>10.4f} {log['test']['metric']:>10.4f}")

### 1.2. CNN 학습

In [ ]:
config_cnn = {
    "dataset_dir": DATASET_DIR,
    "task":        TASK,
    "model":       "cnn",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  10,
    "lr":          0.01,
}

exp_cnn = Experiment(config_cnn)
print(f"CNN 파라미터 수: {sum(p.size for p in exp_cnn.model.params):,}")
print("학습 시작 (10 epochs)...")
logs_cnn = exp_cnn.run()

print()
print(f"{'epoch':>5} | {'train loss':>10} {'train acc':>10} | {'test loss':>10} {'test acc':>10}")
print("-" * 55)
for log in logs_cnn:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>10.4f} {log['train']['metric']:>10.4f} | "
          f"{log['test']['loss']:>10.4f} {log['test']['metric']:>10.4f}")

## 2. 학습 곡선 비교

In [ ]:
epochs = list(range(1, 11))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle(f"Multiclass — MLP vs CNN (10 epochs, lr=0.01, batch=64)",
             fontsize=12, fontweight="bold")

for ax, key, title in zip(axes, ["loss", "metric"], ["Cross-Entropy Loss", "Accuracy"]):
    ax.plot(epochs, [l["train"][key] for l in logs_mlp], color="steelblue", linewidth=2,
            marker="o", label="MLP train")
    ax.plot(epochs, [l["test"][key]  for l in logs_mlp], color="steelblue", linewidth=2,
            marker="o", linestyle="--", label="MLP test")
    ax.plot(epochs, [l["train"][key] for l in logs_cnn], color="#E87B4C",   linewidth=2,
            marker="s", label="CNN train")
    ax.plot(epochs, [l["test"][key]  for l in logs_cnn], color="#E87B4C",   linewidth=2,
            marker="s", linestyle="--", label="CNN test")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("epoch")
    ax.set_ylabel(title.lower())
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[1].set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 3. 예측 grid 비교

In [ ]:
test_ds = MnistDataset("test", TASK, dataset_dir=DATASET_DIR)
N_SHOW  = 16
imgs    = np.stack([test_ds[i][0] for i in range(N_SHOW)])
trues   = np.array([test_ds[i][1].argmax() for i in range(N_SHOW)])

preds_mlp = exp_mlp.predictor.predict(imgs)["predictions"]
preds_cnn = exp_cnn.predictor.predict(imgs)["predictions"]

fig, axes = plt.subplots(2, 8, figsize=(14, 5))
fig.suptitle("Multiclass 예측 grid — MLP(위) vs CNN(아래) (10 epochs)",
             fontsize=11, fontweight="bold")

for col in range(8):
    for row, (preds, label) in enumerate([(preds_mlp, "MLP"), (preds_cnn, "CNN")]):
        ax = axes[row][col]
        ax.imshow(imgs[col].reshape(28, 28), cmap="gray")
        color = "green" if trues[col] == preds[col] else "red"
        ax.set_title(f"T:{trues[col]} P:{preds[col]}", fontsize=7, color=color)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(label, fontsize=9)

plt.tight_layout()
plt.show()

mlp_acc = (trues == preds_mlp).mean()
cnn_acc = (trues == preds_cnn).mean()
print(f"MLP 정답률 ({N_SHOW}개): {mlp_acc:.0%}")
print(f"CNN 정답률 ({N_SHOW}개): {cnn_acc:.0%}")

## 4. 정량 비교

In [ ]:
mlp_params = sum(p.size for p in exp_mlp.model.params)
cnn_params = sum(p.size for p in exp_cnn.model.params)

mlp_final = logs_mlp[-1]["test"]
cnn_final = logs_cnn[-1]["test"]

print("[ Multiclass 최종 결과 비교 ]")
print()
print(f"{'항목':<20} {'MLP':>12} {'CNN':>12}")
print("-" * 46)
print(f"{'파라미터 수':<20} {mlp_params:>12,} {cnn_params:>12,}")
print(f"{'test loss':<20} {mlp_final['loss']:>12.4f} {cnn_final['loss']:>12.4f}")
print(f"{'test accuracy':<20} {mlp_final['metric']:>12.4f} {cnn_final['metric']:>12.4f}")
print(f"{'accuracy 향상':<20} {'':>12} {cnn_final['metric'] - mlp_final['metric']:>+12.4f}")

print()
print("[ Phase 7.2 기준 결과 (저장된 checkpoint 평가) ]")
print(f"  MLP: loss=0.4499, accuracy=0.8839")
print(f"  CNN: loss=0.2106, accuracy=0.9345")

## 5. Checkpoint 저장 및 재평가

In [ ]:
# 저장된 outputs/ checkpoint 불러와 평가 재현
from src.core.evaluator import Evaluator
from src.models.mlp import MLP
from src.models.cnn import CNN
from src.data.dataloader import DataLoader

spec = get_task_spec(TASK)
test_ds  = MnistDataset("test", TASK, dataset_dir=DATASET_DIR)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

results_saved = {}

for model_name, ModelClass in [("mlp", MLP), ("cnn", CNN)]:
    ck_path = os.path.join(OUTPUT_DIR, model_name, "model")
    ck_file = ck_path + ".npz"

    if os.path.isfile(ck_file):
        model = ModelClass(task=TASK, seed=42)
        checkpoints.load(model, ck_path)
        evaluator = Evaluator(model, spec)
        result = evaluator.evaluate(test_loader)
        results_saved[model_name] = result
        print(f"{model_name.upper()} (저장된 checkpoint):  "
              f"loss={result['loss']:.4f}, accuracy={result['metric']:.4f}, "
              f"samples={result['num_samples']}")
    else:
        print(f"{model_name.upper()}: checkpoint 없음 ({ck_file})")

## 6. 저장된 결과 이미지 확인

In [ ]:
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle("outputs/multiclass/ — 저장된 결과 이미지", fontsize=12, fontweight="bold")

for row, model_name in enumerate(["mlp", "cnn"]):
    for col, fname in enumerate(["training_log.png", "predictions.png"]):
        ax = axes[row][col]
        fpath = os.path.join(OUTPUT_DIR, model_name, fname)
        if os.path.isfile(fpath):
            img = Image.open(fpath)
            ax.imshow(np.asarray(img))
            ax.set_title(f"{model_name.upper()} — {fname}", fontsize=9)
        else:
            ax.text(0.5, 0.5, "파일 없음", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(f"{model_name.upper()} — {fname}", fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.show()

## 7. CLI 실행 명령

In [ ]:
print("[ Multiclass 실험 CLI 실행 명령 ]")
print()
for model, env in [("mlp", "numpy_py311"), ("cnn", "cupy_py311_cuda121")]:
    print(f"# {model.upper()} ({env})")
    print(f"PYTHONPATH=. conda run -n {env} python scripts/train.py \\\ ")
    print(f"    --task multiclass --model {model} \\\ ")
    print(f"    --checkpoint outputs/multiclass/{model}/model.npz")
    print()
    print(f"PYTHONPATH=. conda run -n {env} python scripts/evaluate.py \\\ ")
    print(f"    --task multiclass --model {model} \\\ ")
    print(f"    --checkpoint outputs/multiclass/{model}/model.npz")
    print()
    print(f"PYTHONPATH=. MPLBACKEND=Agg conda run -n {env} python scripts/visualize.py \\\ ")
    print(f"    --task multiclass --model {model} \\\ ")
    print(f"    --output_dir outputs/multiclass/{model}")
    print()

## 8. 정리

**Multiclass 과제 규약**

| 항목 | 값 |
|---|---|
| target 변환 | digit → one-hot `(10,)` |
| output_dim | 10 |
| loss | cross_entropy (softmax 내부 처리) |
| metric | accuracy |
| prediction_mode | argmax |

**실험 결과 요약 (Phase 7.2 기준)**

| 모델 | 파라미터 | test loss | test accuracy |
|---|---|---|---|
| MLP | ~300K | 0.4499 | 0.8839 |
| CNN | ~824K | 0.2106 | 0.9345 |

**CNN 우위 이유**
- Conv2d가 이미지의 공간적 패턴(엣지, 획)을 직접 학습
- MLP는 784개 픽셀을 독립 특징으로 처리 — 공간 구조 정보 손실

**후속 프레임워크 연계**
- `config["task"]`, `config["model"]` 키와 CLI 인자 이름을 그대로 유지
- `Trainer.fit()` / `Evaluator.evaluate()` 반환 dict 구조 동일하게 유지